In [28]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

UPLOAD_PATH = "raw_loan_dataset.csv"

df = pd.read_csv(UPLOAD_PATH)

In [29]:
df.head(10)

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810,537.0,1.1,25800,Yes,No,No
1,96482,524.0,15.0,11200,Y,No,Yes
2,28478,NaN,5.4,12100,No,No,Yes
3,"$25,851",561.0,17.6,7000,Yes,No,Yes
4,38396,527.0,9.8,10700,No,No,Approved
5,NaN,754.0,3.6,47800,Yes,No,Yes
6,106070,665.0,14.6,48000,N,No,Yes
7,35458,599.0,21.7,"$10,300",No,No,Yes
8,73520,661.0,NaN,17200,Yes,No,Yes
9,57087,563.0,11.8,13400,Yes,No,Yes


In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     str    
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     str    
 4   HasCollateral     101 non-null    str    
 5   PreviousDefaults  101 non-null    str    
 6   Approved          103 non-null    str    
dtypes: float64(2), str(5)
memory usage: 5.8 KB


In [31]:
print(df.isnull().sum())

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Approved            0
dtype: int64


### clean MONEY formatting

In [32]:
df["Income"] = df["Income"].replace(r"[\$,]", "", regex=True).astype(float)

df["LoanAmount"] = df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)

In [33]:
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810.0,537.0,1.1,25800.0,Yes,No,No
1,96482.0,524.0,15.0,11200.0,Y,No,Yes
2,28478.0,NaN,5.4,12100.0,No,No,Yes
3,25851.0,561.0,17.6,7000.0,Yes,No,Yes
4,38396.0,527.0,9.8,10700.0,No,No,Approved


In [34]:
yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
}

In [35]:
df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})

df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})

df["Approved"] = df["Approved"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})

In [36]:
df.head(10)

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810.0,537.0,1.1,25800.0,Yes,No,No
1,96482.0,524.0,15.0,11200.0,Yes,No,Yes
2,28478.0,NaN,5.4,12100.0,No,No,Yes
3,25851.0,561.0,17.6,7000.0,Yes,No,Yes
4,38396.0,527.0,9.8,10700.0,No,No,Yes
5,NaN,754.0,3.6,47800.0,Yes,No,Yes
6,106070.0,665.0,14.6,48000.0,No,No,Yes
7,35458.0,599.0,21.7,10300.0,No,No,Yes
8,73520.0,661.0,NaN,17200.0,Yes,No,Yes
9,57087.0,563.0,11.8,13400.0,Yes,No,Yes


### immute missing values

In [37]:
df["Income"] = df["Income"].fillna(df["Income"].median())

df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())

df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())

df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())

df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])

df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])

In [38]:
before = df.shape[0]

### remove duplicates

In [39]:
df = df.drop_duplicates()
print(f"\nDropped duplicates: {before} -> {df.shape[0]} rows")


Dropped duplicates: 103 -> 100 rows


### IQR Capping

In [40]:
def capIQR(dataframe, column_name, multiplier = 1.5 ):

    # 1. Calculate Q1, Q3, and IQR
    q1 = dataframe[column_name].quantile(0.25)
    q3 = dataframe[column_name].quantile(0.75)
    iqr = q3 - q1

    # 2. Calculate upper and lower fences
    lower_limit = q1 - (multiplier * iqr)
    upper_limit = q3 + (multiplier * iqr)

    return lower_limit, upper_limit



In [41]:
low_income,  high_income  = capIQR(df, "Income")
low_credit,  high_credit  = capIQR(df, "CreditScore")
low_loan,    high_loan    = capIQR(df, "LoanAmount")
low_emp,     high_emp     = capIQR(df, "EmploymentYears")

In [42]:
df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)

In [43]:
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810.0,537.0,1.1,25800.0,Yes,No,No
1,96482.0,524.0,15.0,11200.0,Yes,No,Yes
2,28478.0,638.0,5.4,12100.0,No,No,Yes
3,25851.0,561.0,17.6,7000.0,Yes,No,Yes
4,38396.0,527.0,9.8,10700.0,No,No,Yes


### label encoding

In [44]:
df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)

### class distribution

In [45]:
print(df["Approved"].value_counts())
print(df["Approved"].value_counts(normalize=True).round(3))

Approved
1    66
0    34
Name: count, dtype: int64
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


### class balance check

In [46]:
class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes — consider balancing techniques (L3).")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")



Class balance OK for baseline Accuracy (both classes well represented).


### feature engineering

In [47]:
df["DebtToIncome"] = df["LoanAmount"] / df["Income"].replace(0, np.nan)
df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)

In [48]:
binary_cols = {"HasCollateral", "PreviousDefaults", "Approved"}
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
scale_cols = [c for c in numeric_cols if c not in binary_cols]

### RobustScaler

In [50]:
scaler = RobustScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

## finals

In [51]:
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved,DebtToIncome,IncomePerYearEmployed
0,0.822149,-0.653722,-0.978632,0.246080,Yes,No,0,-0.445244,8.274536
1,0.564574,-0.737864,0.209402,-0.458384,Yes,No,1,-0.912749,0.153994
2,-0.856268,0.000000,-0.611111,-0.414958,No,No,1,0.280113,-0.126321
3,-0.911156,-0.498382,0.431624,-0.661037,Yes,No,1,-0.315175,-0.669033
4,-0.649046,-0.718447,-0.235043,-0.482509,No,No,1,-0.284688,-0.284975


In [52]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 100 non-null    float64
 1   CreditScore            100 non-null    float64
 2   EmploymentYears        100 non-null    float64
 3   LoanAmount             100 non-null    float64
 4   HasCollateral          100 non-null    str    
 5   PreviousDefaults       100 non-null    str    
 6   Approved               100 non-null    int64  
 7   DebtToIncome           100 non-null    float64
 8   IncomePerYearEmployed  100 non-null    float64
dtypes: float64(6), int64(1), str(2)
memory usage: 7.2 KB


In [53]:
df.isnull().sum()

Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Approved                 0
DebtToIncome             0
IncomePerYearEmployed    0
dtype: int64

In [ ]:
EXPORT_AS = "raw_loan_dataset.csv"

df.to_csv(EXPORT_AS)